In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/sml project/datasets/Final_Augmented_dataset_Diseases_and_Symptoms.csv')

### Data Cleaning and Preprocessing for Model Training

Given that some disease classes might have very few instances, we'll first analyze the distribution of the 'diseases' column. Classes with very low counts can cause issues during model training (e.g., a model might not learn patterns for them or they might not appear in both training and test sets). We'll also check for any missing values and then prepare the dataset by separating features and the target variable, and encoding the target.

In [3]:
# 1. Analyze 'diseases' column distribution
disease_counts = df['diseases'].value_counts()
display(disease_counts.head(10))
display(disease_counts.tail(10))

# Identify diseases with low frequency (e.g., less than 5 samples)
rare_diseases = disease_counts[disease_counts < 80].index
print(f"\nNumber of rare diseases (less than 5 samples): {len(rare_diseases)}")
print("Rare diseases to be removed or grouped:", list(rare_diseases))

,count
diseases,
cystitis,1219
vulvodynia,1218
nose disorder,1218
complex regional pain syndrome,1217
spondylosis,1216
esophagitis,1215
conjunctivitis due to allergy,1215
hypoglycemia,1215
peripheral nerve disorder,1215


,count
diseases,
heat stroke,1
chronic ulcer,1
open wound of the knee,1
open wound of the chest,1
open wound due to trauma,1
thalassemia,1
huntington disease,1
typhoid fever,1
kaposi sarcoma,1



Number of rare diseases (less than 5 samples): 298
Rare diseases to be removed or grouped: ['somatization disorder', 'fracture of the patella', 'toxic multinodular goiter', 'lewy body dementia', 'esophageal cancer', 'vaginismus', 'chorioretinitis', 'menopause', 'goiter', 'epilepsy', 'fracture of the jaw', 'parasitic disease', 'hypernatremia', 'hypothyroidism', 'adhesive capsulitis of the shoulder', 'ascending cholangitis', 'pityriasis rosea', 'atonic bladder', 'aplastic anemia', 'central retinal artery or vein occlusion', 'atrial fibrillation', 'ovarian torsion', 'poisoning due to analgesics', 'lice', 'fracture of the skull', 'protein deficiency', 'dislocation of the shoulder', 'injury to the abdomen', 'crushing injury', 'drug abuse (barbiturates)', 'septic arthritis', 'salivary gland disorder', 'nonalcoholic liver disease (nash)', 'hydatidiform mole', 'endophthalmitis', 'testicular disorder', 'amyotrophic lateral sclerosis (als)', 'hematoma', 'phimosis', 'lichen simplex', 'guillain b

From the value counts, we can identify diseases that appear very infrequently. For a robust model, it's often best to either remove these rare classes or group them into an 'Other' category if they are too numerous. For now, we will remove rows corresponding to diseases with less than 5 samples to ensure a minimum representation for each class.

In [4]:
# Strip whitespace and convert to lowercase for both
df['diseases'] = df['diseases'].str.strip().str.lower()
rare_diseases = [d.strip().lower() for d in rare_diseases]

# Now run the filter
df_cleaned = df[~df['diseases'].isin(rare_diseases)]

print(f"Original DataFrame shape: {df.shape}")
print(f"Cleaned DataFrame shape after removing rare diseases: {df_cleaned.shape}")

unique_count = df['diseases'].nunique()
print(f"Number of unique diseases: {unique_count}")

unique_count = df_cleaned['diseases'].nunique()
print(f"Number of unique diseases: {unique_count}")

# Verify the new distribution
display(df_cleaned['diseases'].value_counts().tail(10))

Original DataFrame shape: (246945, 378)
Cleaned DataFrame shape after removing rare diseases: (239444, 378)
Number of unique diseases: 773
Number of unique diseases: 475


,count
diseases,
achalasia,84
bone cancer,83
endocarditis,83
gynecomastia,83
dyshidrosis,83
extrapyramidal effect of drugs,83
spina bifida,82
spermatocele,81
kidney cancer,80


In [5]:
from sklearn.preprocessing import LabelEncoder

# Separate features (X) and target (y)
X = df_cleaned.drop('diseases', axis=1)
y_disease_names = df_cleaned['diseases']

# Encode the target variable
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_disease_names)

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")
print("Encoded target classes:", label_encoder.classes_[:5], '...') # Display first 5 classes
print("Example original disease names:", y_disease_names[:5].tolist())
print("Example encoded labels:", y[:5].tolist())

Shape of features (X): (239444, 377)
Shape of target (y): (239444,)
Encoded target classes: ['abdominal aortic aneurysm' 'abdominal hernia' 'abscess of nose'
 'abscess of the pharynx' 'achalasia'] ...
Example original disease names: ['panic disorder', 'panic disorder', 'panic disorder', 'panic disorder', 'panic disorder']
Example encoded labels: [319, 319, 319, 319, 319]


In [6]:
from sklearn.model_selection import train_test_split

# Step 1: Split off 30% as a temporary holdout (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step 2: Split the 30% holdout evenly into val (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Shape of X_train: {X_train.shape}")   # ~70%
print(f"Shape of X_val:   {X_val.shape}")     # ~15%
print(f"Shape of X_test:  {X_test.shape}")    # ~15%
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val:   {y_val.shape}")
print(f"Shape of y_test:  {y_test.shape}")

Shape of X_train: (167610, 377)
Shape of X_val:   (35917, 377)
Shape of X_test:  (35917, 377)
Shape of y_train: (167610,)
Shape of y_val:   (35917,)
Shape of y_test:  (35917,)


In [6]:
from cuml.linear_model import LogisticRegression as cuLogisticRegression
from sklearn.metrics import classification_report
import cupy as cp
import numpy as np

# Convert all three splits to GPU
X_train_gpu = cp.array(X_train.astype('float32'))
y_train_gpu = cp.array(y_train.astype('float32'))
X_val_gpu   = cp.array(X_val.astype('float32'))
y_val_gpu   = cp.array(y_val.astype('float32'))
X_test_gpu  = cp.array(X_test.astype('float32'))
y_test_gpu  = cp.array(y_test.astype('float32'))

# Train
lr_gpu = cuLogisticRegression(solver='qn', max_iter=1000, tol=1e-4, verbose=True)
lr_gpu.fit(X_train_gpu, y_train_gpu)

def evaluate(model, X, y_gpu, split_name=""):
    y_cpu = y_gpu.get()

    # Top-1 Accuracy
    top1 = model.score(X, y_gpu)

    # Probabilities for top-k
    probs = model.predict_proba(X).get()

    # Top-3 Accuracy
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_cpu.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_cpu.reshape(-1, 1), axis=1))

    # Macro F1
    y_pred = model.predict(X).get()
    report = classification_report(y_cpu, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']

    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")

# Use validation set for tuning decisions
evaluate(lr_gpu, X_val_gpu, y_val_gpu, split_name="Validation")

# Only run this once at the very end for final reported numbers
evaluate(lr_gpu, X_test_gpu, y_test_gpu, split_name="Test")

[2026-04-16 22:02:48.637] [CUML] [debug] Running L-BFGS
[2026-04-16 22:02:48.864] [CUML] [debug] 0000: f(x)=6.16388178 conv.crit=0.00048410 (gnorm=0.00298394, fmag=6.16388178)
[2026-04-16 22:02:48.917] [CUML] [debug] 0001: f(x)=6.06587029 conv.crit=0.00047755 (gnorm=0.00289677, fmag=6.06587029)
[2026-04-16 22:02:48.996] [CUML] [debug] 0002: f(x)=3.61034989 conv.crit=0.00973981 (gnorm=0.03516411, fmag=3.61034989)
[2026-04-16 22:02:49.031] [CUML] [debug] 0003: f(x)=3.37008095 conv.crit=0.00668938 (gnorm=0.02254375, fmag=3.37008095)
[2026-04-16 22:02:49.066] [CUML] [debug] 0004: f(x)=2.30810523 conv.crit=0.00490351 (gnorm=0.01131782, fmag=2.30810523)
[2026-04-16 22:02:49.101] [CUML] [debug] 0005: f(x)=1.55482924 conv.crit=0.00400640 (gnorm=0.00622926, fmag=1.55482924)
[2026-04-16 22:02:49.136] [CUML] [debug] 0006: f(x)=1.17997670 conv.crit=0.01100981 (gnorm=0.01299132, fmag=1.17997670)
[2026-04-16 22:02:49.171] [CUML] [debug] 0007: f(x)=0.89039063 conv.crit=0.00645011 (gnorm=0.00574312, f

In [7]:
# 1. Convert your symptom dataframe (X) to a GPU array
# This assumes X contains only your 0/1 symptom columns
X_gpu = cp.asarray(X.values)
# 2. Compute the Co-occurrence Matrix using Dot Product
# Matrix multiplication: (377x240k) * (240k*377) = (377x377)
# This counts every time symptom i and symptom j appear in the same row
co_matrix_gpu = cp.dot(X_gpu.T, X_gpu)
# 3. Move back to CPU and format as a DataFrame
full_co_matrix = pd.DataFrame(
co_matrix_gpu.get(),
index=X.columns,
columns=X.columns
)
# 4. Set the diagonal to 0 (we don't want to suggest 'fever' because the user said 'fever')
import numpy as np
np.fill_diagonal(full_co_matrix.values, 0)
print(f"Matrix complete. Shape: {full_co_matrix.shape}") # Should be (377, 377)

Matrix complete. Shape: (377, 377)


In [8]:
from cuml.linear_model import LogisticRegression as cuLogisticRegression
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
import cupy as cp
import numpy as np

# Convert all three splits to GPU
X_train_gpu = cp.array(X_train.astype('float32'))
y_train_gpu = cp.array(y_train.astype('float32'))
X_val_gpu   = cp.array(X_val.astype('float32'))
y_val_gpu   = cp.array(y_val.astype('float32'))
X_test_gpu  = cp.array(X_test.astype('float32'))
y_test_gpu  = cp.array(y_test.astype('float32'))

# Class weights to handle imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
sample_weights_gpu = cp.array(sample_weights.astype('float32'))

# Train with tuned hyperparameters
lr_gpu = cuLogisticRegression(solver='qn', max_iter=2000, tol=1e-5, C=0.5, verbose=True)
lr_gpu.fit(X_train_gpu, y_train_gpu, sample_weight=sample_weights_gpu)

# Knowledge graph feature expansion
def expand_features(X_df, co_matrix, top_n=3):
    expanded = X_df.copy()
    for symptom in X_df.columns:
        neighbors = co_matrix[symptom].nlargest(top_n).index
        expanded[f"{symptom}_neighbors"] = X_df[neighbors].max(axis=1)
    return expanded

X_train_exp = expand_features(X_train, full_co_matrix)
X_val_exp   = expand_features(X_val,   full_co_matrix)
X_test_exp  = expand_features(X_test,  full_co_matrix)

X_train_exp_gpu = cp.array(X_train_exp.astype('float32'))
X_val_exp_gpu   = cp.array(X_val_exp.astype('float32'))
X_test_exp_gpu  = cp.array(X_test_exp.astype('float32'))

# Train expanded model
lr_exp = cuLogisticRegression(solver='qn', max_iter=2000, tol=1e-5, C=0.5, verbose=True)
lr_exp.fit(X_train_exp_gpu, y_train_gpu, sample_weight=sample_weights_gpu)

def evaluate(model, X, y_gpu, split_name=""):
    y_cpu = y_gpu.get()
    top1  = model.score(X, y_gpu)
    probs = model.predict_proba(X).get()

    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_cpu.reshape(-1, 1), axis=1))

    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_cpu.reshape(-1, 1), axis=1))

    y_pred   = model.predict(X).get()
    macro_f1 = classification_report(y_cpu, y_pred, output_dict=True, zero_division=0)['macro avg']['f1-score']

    print(f"\n--- {split_name} ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")

# Compare base vs expanded on validation
print("=== Base Model ===")
evaluate(lr_gpu, X_val_gpu, y_val_gpu, "Validation")

print("\n=== Expanded Model (+ Knowledge Graph Features) ===")
evaluate(lr_exp, X_val_exp_gpu, y_val_gpu, "Validation")

# Ensemble on validation
probs_base = lr_gpu.predict_proba(X_val_gpu).get()
probs_exp  = lr_exp.predict_proba(X_val_exp_gpu).get()
ens_probs  = 0.5 * probs_base + 0.5 * probs_exp
y_val_cpu  = y_val_gpu.get()

top1_ens = np.mean(np.argmax(ens_probs, axis=1) == y_val_cpu)
top3_ens = np.mean(np.any(np.argsort(ens_probs, axis=1)[:, -3:] == y_val_cpu.reshape(-1, 1), axis=1))
top5_ens = np.mean(np.any(np.argsort(ens_probs, axis=1)[:, -5:] == y_val_cpu.reshape(-1, 1), axis=1))

print("\n=== Ensemble (Base + Expanded) — Validation ===")
print(f"Top-1 Accuracy: {top1_ens:.4f}")
print(f"Top-3 Accuracy: {top3_ens:.4f}")
print(f"Top-5 Accuracy: {top5_ens:.4f}")

# --- Run test evaluation only once on your best model ---
# Uncomment whichever performs best on validation above
evaluate(lr_gpu,  X_test_gpu,     y_test_gpu, "Test — Base Model")
evaluate(lr_exp,  X_test_exp_gpu, y_test_gpu, "Test — Expanded Model")

[2026-04-16 22:03:21.390] [CUML] [debug] Running L-BFGS
[2026-04-16 22:03:21.443] [CUML] [debug] 0000: f(x)=6.16388178 conv.crit=0.00048410 (gnorm=0.00298394, fmag=6.16388178)
[2026-04-16 22:03:21.496] [CUML] [debug] 0001: f(x)=6.06587315 conv.crit=0.00047755 (gnorm=0.00289677, fmag=6.06587315)
[2026-04-16 22:03:21.602] [CUML] [debug] 0002: f(x)=3.63617754 conv.crit=0.00961659 (gnorm=0.03496764, fmag=3.63617754)
[2026-04-16 22:03:21.653] [CUML] [debug] 0003: f(x)=3.43142939 conv.crit=0.00649914 (gnorm=0.02230134, fmag=3.43142939)
[2026-04-16 22:03:21.688] [CUML] [debug] 0004: f(x)=2.36143732 conv.crit=0.00474795 (gnorm=0.01121198, fmag=2.36143732)
[2026-04-16 22:03:21.721] [CUML] [debug] 0005: f(x)=1.59560192 conv.crit=0.00413919 (gnorm=0.00660450, fmag=1.59560192)
[2026-04-16 22:03:21.756] [CUML] [debug] 0006: f(x)=1.24019074 conv.crit=0.01046271 (gnorm=0.01297576, fmag=1.24019074)
[2026-04-16 22:03:21.790] [CUML] [debug] 0007: f(x)=0.95461440 conv.crit=0.00565454 (gnorm=0.00539791, f

/tmp/ipykernel_16396/3953092848.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  expanded[f"{symptom}_neighbors"] = X_df[neighbors].max(axis=1)
/tmp/ipykernel_16396/3953092848.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  expanded[f"{symptom}_neighbors"] = X_df[neighbors].max(axis=1)
/tmp/ipykernel_16396/3953092848.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat

[2026-04-16 22:03:38.482] [CUML] [debug] Running L-BFGS
[2026-04-16 22:03:38.580] [CUML] [debug] 0000: f(x)=6.16388178 conv.crit=0.00072594 (gnorm=0.00447458, fmag=6.16388178)
[2026-04-16 22:03:38.677] [CUML] [debug] 0001: f(x)=5.86248589 conv.crit=0.00075707 (gnorm=0.00443834, fmag=5.86248589)
[2026-04-16 22:03:38.753] [CUML] [debug] 0002: f(x)=4.75600863 conv.crit=0.01078393 (gnorm=0.05128848, fmag=4.75600863)
[2026-04-16 22:03:38.815] [CUML] [debug] 0003: f(x)=4.49362326 conv.crit=0.00410403 (gnorm=0.01844198, fmag=4.49362326)
[2026-04-16 22:03:38.876] [CUML] [debug] 0004: f(x)=3.50144315 conv.crit=0.00309753 (gnorm=0.01084582, fmag=3.50144315)
[2026-04-16 22:03:38.938] [CUML] [debug] 0005: f(x)=3.08232951 conv.crit=0.00394356 (gnorm=0.01215536, fmag=3.08232951)
[2026-04-16 22:03:39.000] [CUML] [debug] 0006: f(x)=2.62063193 conv.crit=0.00265948 (gnorm=0.00696951, fmag=2.62063193)
[2026-04-16 22:03:39.062] [CUML] [debug] 0007: f(x)=2.39437485 conv.crit=0.00262255 (gnorm=0.00627938, f

In [10]:
import xgboost as xgb
import cupy as cp
import numpy as np
from sklearn.metrics import classification_report

# Convert all three splits to GPU for XGBoost
# Ensure y is int32 for XGBoost
X_train_gpu_xgb = cp.array(X_train.astype('float32'))
y_train_gpu_xgb = cp.array(y_train.astype('int32'))
X_val_gpu_xgb   = cp.array(X_val.astype('float32'))
y_val_gpu_xgb   = cp.array(y_val.astype('int32'))
X_test_gpu_xgb  = cp.array(X_test.astype('float32'))
y_test_gpu_xgb  = cp.array(y_test.astype('int32'))

def evaluate_xgb(model, X_gpu, y_gpu, split_name=""):
    y_cpu = y_gpu.get()

    # Safely get best iteration if early stopping triggered
    best_iter = model.best_iteration if hasattr(model, 'best_iteration') else None

    # Get Probabilities (for Top-1, Top-3, Top-5) using predict_proba!
    if best_iter is not None:
        probs = model.predict_proba(X_gpu, iteration_range=(0, best_iter + 1))
        y_pred = model.predict(X_gpu, iteration_range=(0, best_iter + 1))
    else:
        probs = model.predict_proba(X_gpu)
        y_pred = model.predict(X_gpu)

    # If using CuPy arrays for inference, bring results back to CPU
    if hasattr(probs, 'get'): probs = probs.get()
    if hasattr(y_pred, 'get'): y_pred = y_pred.get()

    # Top-1 Accuracy
    y_pred_top1 = np.argmax(probs, axis=1)
    top1 = np.mean(y_pred_top1 == y_cpu)

    # Top-3 Accuracy
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_cpu.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_cpu.reshape(-1, 1), axis=1))

    # Macro F1
    report = classification_report(y_cpu, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']

    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")

# Initialize XGBClassifier with GPU device and early stopping
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=1,
    tree_method='hist',
    device='cuda',             # CRITICAL: Forces execution on the NVIDIA GPU
    early_stopping_rounds=20,  # Put this here for modern XGBoost versions
    eval_metric='mlogloss',
    random_state=42
)

# Train the model
print("Training XGBoost...")
xgb_model.fit(
    X_train_gpu_xgb, y_train_gpu_xgb,
    eval_set=[(X_val_gpu_xgb, y_val_gpu_xgb)],
    verbose=10 # Set to 10 so it prints every 10 trees instead of every single one
)

print(f"\nTraining stopped. Best iteration: {xgb_model.best_iteration}")

# Evaluate on validation set
evaluate_xgb(xgb_model, X_val_gpu_xgb, y_val_gpu_xgb, "XGBoost Validation")

# Evaluate on test set
evaluate_xgb(xgb_model, X_test_gpu_xgb, y_test_gpu_xgb, "XGBoost Test")

Training XGBoost...
[0]	validation_0-mlogloss:3.30463
[10]	validation_0-mlogloss:1.30265
[20]	validation_0-mlogloss:0.84467
[30]	validation_0-mlogloss:0.64107
[40]	validation_0-mlogloss:0.53889
[50]	validation_0-mlogloss:0.48483
[60]	validation_0-mlogloss:0.45552
[70]	validation_0-mlogloss:0.44005
[80]	validation_0-mlogloss:0.43214
[90]	validation_0-mlogloss:0.42885
[100]	validation_0-mlogloss:0.42837
[110]	validation_0-mlogloss:0.42993
[114]	validation_0-mlogloss:0.43071

Training stopped. Best iteration: 94

--- XGBoost Validation Results ---
Top-1 Accuracy: 0.8420
Top-3 Accuracy: 0.9529
Top-5 Accuracy: 0.9765
Macro F1 Score: 0.8559

--- XGBoost Test Results ---
Top-1 Accuracy: 0.8443
Top-3 Accuracy: 0.9516
Top-5 Accuracy: 0.9768
Macro F1 Score: 0.8578


In [11]:
def evaluate_xgb_1(model, X_gpu, y_gpu, split_name=""):
    y_cpu = y_gpu.get()

    # Safely get best iteration if early stopping triggered
    best_iter = model.best_iteration if hasattr(model, 'best_iteration') else None

    # Get Probabilities (for Top-1, Top-3, Top-5) using predict_proba!
    if best_iter is not None:
        probs = model.predict_proba(X_gpu, iteration_range=(0, best_iter + 1))
        y_pred = model.predict(X_gpu, iteration_range=(0, best_iter + 1))
    else:
        probs = model.predict_proba(X_gpu)
        y_pred = model.predict(X_gpu)

    # If using CuPy arrays for inference, bring results back to CPU
    if hasattr(probs, 'get'): probs = probs.get()
    if hasattr(y_pred, 'get'): y_pred = y_pred.get()

    # Top-1 Accuracy
    y_pred_top1 = np.argmax(probs, axis=1)
    top1 = np.mean(y_pred_top1 == y_cpu)

    # Top-3 Accuracy
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_cpu.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_cpu.reshape(-1, 1), axis=1))

    # F1 Scores (Macro and Weighted)
    report = classification_report(y_cpu, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score'] # <-- Added Weighted F1

    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}")

In [12]:
evaluate_xgb_1(xgb_model, X_val_gpu_xgb, y_val_gpu_xgb, "XGBoost Validation")

# Evaluate on test set
evaluate_xgb_1(xgb_model, X_test_gpu_xgb, y_test_gpu_xgb, "XGBoost Test")


--- XGBoost Validation Results ---
Top-1 Accuracy: 0.8420
Top-3 Accuracy: 0.9529
Top-5 Accuracy: 0.9765
Macro F1 Score: 0.8559
Weighted F1 Score: 0.8424

--- XGBoost Test Results ---
Top-1 Accuracy: 0.8443
Top-3 Accuracy: 0.9516
Top-5 Accuracy: 0.9768
Macro F1 Score: 0.8578
Weighted F1 Score: 0.8450


In [13]:
def evaluate(model, X, y_gpu, split_name=""):
    y_cpu = y_gpu.get()

    # Top-1 Accuracy
    top1 = model.score(X, y_gpu)

    # Probabilities for top-k
    probs = model.predict_proba(X).get()

    # Top-3 Accuracy
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_cpu.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_cpu.reshape(-1, 1), axis=1))

    # F1 Scores (Macro and Weighted)
    y_pred = model.predict(X).get()
    report = classification_report(y_cpu, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score'] # <-- Added Weighted F1

    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}") # <-- Added Print Statement

In [14]:
# 1. Evaluate on the Validation set (Use this while tuning hyperparameters)
evaluate(lr_gpu, X_val_gpu, y_val_gpu, split_name="Logistic Regression Validation")

# 2. Evaluate on the Test set (Run this for your final project metrics)
evaluate(lr_gpu, X_test_gpu, y_test_gpu, split_name="Logistic Regression Test")


--- Logistic Regression Validation Results ---
Top-1 Accuracy: 0.8675
Top-3 Accuracy: 0.9646
Top-5 Accuracy: 0.9830
Macro F1 Score: 0.8782
Weighted F1 Score: 0.8678

--- Logistic Regression Test Results ---
Top-1 Accuracy: 0.8690
Top-3 Accuracy: 0.9618
Top-5 Accuracy: 0.9824
Macro F1 Score: 0.8789
Weighted F1 Score: 0.8695


In [15]:
# Optional: Check performance on the training data itself
evaluate(lr_gpu, X_train_gpu, y_train_gpu, split_name="Logistic Regression Training")


--- Logistic Regression Training Results ---
Top-1 Accuracy: 0.8773
Top-3 Accuracy: 0.9681
Top-5 Accuracy: 0.9855
Macro F1 Score: 0.8886
Weighted F1 Score: 0.8776


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Initialize the Random Forest Classifier
# Using a small number of estimators for faster initial training
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training Random Forest Classifier...")
# Ensure we are using the exact same X_train and y_train as the other models
rf_model.fit(X_train, y_train)
print("Random Forest Classifier training complete.")

# Define a consistent evaluation function
def evaluate_rf(model, X_eval, y_eval, split_name=""):
    # The original code had: `if hasattr(y_eval, 'get'): y_eval = y_eval.get()`
    # and `if hasattr(X_eval, 'get'): X_eval = X_eval.get()`
    # These lines are for converting CuPy arrays to NumPy arrays.
    # However, X_eval and y_eval for RandomForest are pandas DataFrames/NumPy arrays,
    # so calling .get() on them causes a TypeError (pandas.DataFrame.get() requires a key).
    # They are removed as they are not needed for sklearn models.

    # Get predictions and probabilities
    y_pred = model.predict(X_eval)
    probs = model.predict_proba(X_eval)

    # Top-1 Accuracy
    top1 = accuracy_score(y_eval, y_pred)

    # Top-3 Accuracy
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_eval.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_eval.reshape(-1, 1), axis=1))

    # F1 Scores (Macro and Weighted)
    report = classification_report(y_eval, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']

    # Print Results
    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}")

# --- Evaluate on all three splits ---
evaluate_rf(rf_model, X_train, y_train, "Random Forest Training")
evaluate_rf(rf_model, X_val, y_val, "Random Forest Validation")
evaluate_rf(rf_model, X_test, y_test, "Random Forest Test")

Training Random Forest Classifier...
Random Forest Classifier training complete.

--- Random Forest Training Results ---
Top-1 Accuracy: 0.9311
Top-3 Accuracy: 0.9919
Top-5 Accuracy: 0.9980
Macro F1 Score: 0.9319
Weighted F1 Score: 0.9314

--- Random Forest Validation Results ---
Top-1 Accuracy: 0.8386
Top-3 Accuracy: 0.9478
Top-5 Accuracy: 0.9700
Macro F1 Score: 0.8529
Weighted F1 Score: 0.8390

--- Random Forest Test Results ---
Top-1 Accuracy: 0.8408
Top-3 Accuracy: 0.9469
Top-5 Accuracy: 0.9681
Macro F1 Score: 0.8565
Weighted F1 Score: 0.8411


In [9]:
from cuml.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
import cupy as cp
import numpy as np

# Convert all three splits to GPU
X_train_gpu = cp.array(X_train.astype('float32'))
y_train_gpu = cp.array(y_train.astype('float32'))
X_val_gpu   = cp.array(X_val.astype('float32'))
y_val_gpu   = cp.array(y_val.astype('float32'))
X_test_gpu  = cp.array(X_test.astype('float32'))
y_test_gpu  = cp.array(y_test.astype('float32'))

# Initialize and train the GPU-accelerated Naive Bayes model
nb_gpu = GaussianNB()

print("Training Naive Bayes Classifier...")
nb_gpu.fit(X_train_gpu, y_train_gpu)
print("Naive Bayes Classifier training complete.")

# Define a consistent evaluation function
def evaluate_nb(model, X_gpu, y_gpu, split_name=""):
    # Ensure true labels are on the CPU for standard evaluation metrics
    y_cpu = y_gpu.get()

    # Get predictions and probabilities from the GPU, then move them to CPU
    probs = model.predict_proba(X_gpu).get()
    y_pred = model.predict(X_gpu).get()

    # Top-1 Accuracy
    top1 = accuracy_score(y_cpu, y_pred)

    # Top-3 Accuracy (Optimized vectorized approach)
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_cpu.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_cpu.reshape(-1, 1), axis=1))

    # F1 Scores (Macro and Weighted)
    report = classification_report(y_cpu, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']

    # Print Results
    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}")

# --- Evaluate on all three splits ---
evaluate_nb(nb_gpu, X_train_gpu, y_train_gpu, "Naive Bayes Training")
evaluate_nb(nb_gpu, X_val_gpu, y_val_gpu, "Naive Bayes Validation")
evaluate_nb(nb_gpu, X_test_gpu, y_test_gpu, "Naive Bayes Test")

Training Naive Bayes Classifier...
Naive Bayes Classifier training complete.

--- Naive Bayes Training Results ---
Top-1 Accuracy: 0.8690
Top-3 Accuracy: 0.9641
Top-5 Accuracy: 0.9828
Macro F1 Score: 0.8683
Weighted F1 Score: 0.8711

--- Naive Bayes Validation Results ---
Top-1 Accuracy: 0.8650
Top-3 Accuracy: 0.9631
Top-5 Accuracy: 0.9824
Macro F1 Score: 0.8651
Weighted F1 Score: 0.8670

--- Naive Bayes Test Results ---
Top-1 Accuracy: 0.8659
Top-3 Accuracy: 0.9616
Top-5 Accuracy: 0.9818
Macro F1 Score: 0.8648
Weighted F1 Score: 0.8683


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# 1. Convert CuPy arrays back to standard NumPy arrays
# TensorFlow has its own internal GPU memory management, so we pass it standard
# NumPy arrays and it handles the GPU acceleration under the hood.
X_train_np = X_train_gpu.get() if hasattr(X_train_gpu, 'get') else X_train_gpu
y_train_np = y_train_gpu.get() if hasattr(y_train_gpu, 'get') else y_train_gpu
X_val_np   = X_val_gpu.get() if hasattr(X_val_gpu, 'get') else X_val_gpu
y_val_np   = y_val_gpu.get() if hasattr(y_val_gpu, 'get') else y_val_gpu
X_test_np  = X_test_gpu.get() if hasattr(X_test_gpu, 'get') else X_test_gpu
y_test_np  = y_test_gpu.get() if hasattr(y_test_gpu, 'get') else y_test_gpu

num_features = X_train_np.shape[1] # 377 Symptoms
num_classes = len(np.unique(y_train_np)) # 721 Diseases

# 2. Build the Multi-Layer Perceptron (MLP) Neural Network
mlp_model = Sequential([
    Dense(512, activation='relu', input_shape=(num_features,)),
    Dropout(0.3), # Drops 30% of neurons randomly to prevent overfitting on sparse data
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax') # Outputs probabilities across all 721 classes
])

# Compile the model
mlp_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Used when labels are integers
    metrics=['accuracy']
)

print("Training Deep Learning MLP Model on GPU...")

# 3. Train the model
# Deep learning requires multiple passes over the data (epochs)
history = mlp_model.fit(
    X_train_np, y_train_np,
    validation_data=(X_val_np, y_val_np),
    epochs=15,
    batch_size=512,     # Large batch size to fully utilize your A100 GPU
    verbose=1
)

# 4. Consistent Evaluation Function
def evaluate_mlp(model, X_eval, y_eval, split_name=""):

    # In Keras, .predict() directly returns the probability matrix
    probs = model.predict(X_eval, verbose=0)

    # Get standard top-1 predictions
    y_pred = np.argmax(probs, axis=1)

    # Top-1 Accuracy
    top1 = accuracy_score(y_eval, y_pred)

    # Top-3 Accuracy
    top3_preds = np.argsort(probs, axis=1)[:, -3:]
    top3 = np.mean(np.any(top3_preds == y_eval.reshape(-1, 1), axis=1))

    # Top-5 Accuracy
    top5_preds = np.argsort(probs, axis=1)[:, -5:]
    top5 = np.mean(np.any(top5_preds == y_eval.reshape(-1, 1), axis=1))

    # F1 Scores
    report = classification_report(y_eval, y_pred, output_dict=True, zero_division=0)
    macro_f1 = report['macro avg']['f1-score']
    weighted_f1 = report['weighted avg']['f1-score']

    print(f"\n--- {split_name} Results ---")
    print(f"Top-1 Accuracy: {top1:.4f}")
    print(f"Top-3 Accuracy: {top3:.4f}")
    print(f"Top-5 Accuracy: {top5:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Weighted F1 Score: {weighted_f1:.4f}")

# 5. Evaluate on all splits to check for overfitting
evaluate_mlp(mlp_model, X_train_np, y_train_np, "Deep Learning MLP Training")
evaluate_mlp(mlp_model, X_val_np, y_val_np, "Deep Learning MLP Validation")
evaluate_mlp(mlp_model, X_test_np, y_test_np, "Deep Learning MLP Test")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Deep Learning MLP Model on GPU...
Epoch 1/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.5850 - loss: 1.9704 - val_accuracy: 0.8452 - val_loss: 0.4661
Epoch 2/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8211 - loss: 0.5461 - val_accuracy: 0.8574 - val_loss: 0.3736
Epoch 3/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8380 - loss: 0.4660 - val_accuracy: 0.8599 - val_loss: 0.3534
Epoch 4/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8446 - loss: 0.4349 - val_accuracy: 0.8600 - val_loss: 0.3480
Epoch 5/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8495 - loss: 0.4129 - val_accuracy: 0.8604 - val_loss: 0.3432
Epoch 6/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8514 - loss: 0.4015 - val_accuracy: 0.8607 - val_loss: 0.3426
Epoch 7/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8540 - loss: 0.3915 - val_accuracy: 0.8613 - val_loss: 0.3404
Epoch 8/15
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accura